# C1DB1B: Data Management for Decision-Making
## Workshop 1: Decision Framing, KPI-to-Data Mapping & Baseline Demonstration
### Executive decision: Where should Vårdlyftet establish its next primary care centre?

---

### The assignment you are supporting
Vårdlyftet AB is considering a primary care centre in Västra Götaland. Your group has been allocated **five municipalities**. The management question, from Document A (*The Assignment*), is:

> Which municipality in your portfolio should Vårdlyftet choose for its next primary care centre, **if any**, and why?

You are advising Vårdlyftet. You are **not** making a regional policy decision on behalf of Västra Götalandsregionen.

In **Module 1 you do not have to pick a winner**. You must:
1. state what the decision is and what should matter;
2. connect proposed indicators to actual data fields;
3. calculate three prescribed demographic indicators for your five municipalities;
4. record an initial weight profile;
5. show that you can explain what happens when a controlled input changes.

Keep this question in view throughout the notebook:

> How does the work we are doing now help us decide where Vårdlyftet should establish its next primary care centre?

---

### Sequence you will reuse in every later module

**Decision** → **Criterion** → **Indicator** → **Data** → **Calculation** → **Interpretation**

The issued anchor and the three demographic calculations are the **worked method**. They are not your finished architecture. After you can run that sequence, your group maps **its own** KPIs for the allocated portfolio, collects the extracts those KPIs need, and carries that set through Modules 2–4. Modify the starter code where your sources require it.

---

### Data vintage (do not hard-code years)
This notebook reads **vintage, file path, and reference years** from `data/manifest.json`.

Current contract (Document B, *The Data Stream*):
- Anchor file: `data/01_module1/VGR_Demographics_Anchor.csv`
- Grain: one row = one municipality $\times$ one year
- Reference years $t_0$ and $t_1$ are defined in the manifest, not in this cell.

If the course team issues a later SCB extract, only `data/` and the manifest change. Your formulas stay the same.


---
### Who does what

**You, before the workshop:** bring a **laptop** with this notebook already able to run. Replace the example municipality codes with your allocated five, complete the three demographic calculations, and draft the KPI-to-data table for **at least six** scored indicators. For every indicator that is not already in the anchor file, name the table you will download. Formatting and joining that extract is your work; there is no source-specific support. Arrive with outputs, not an empty file.

**The teacher, in the lecture:** explains the decision sequence and the live-change pattern.

**The teacher, in the workshop:** checks the common anchor, releases one controlled change, and may ask any member to explain.

**You, in the workshop:** freeze the baseline, write the prediction, apply the change on a **copy**, and explain.

The five alternatives are **municipalities**, not clinic sites. A solution notebook for the demonstration portfolio will be released after the workshop. Do not copy another group's municipality choice.


---
## Part 1: Load the data contract and the Module 1 anchor

The working table already uses the handbook field names:

`Municipality_Code`, `Municipality_Name`, `Year`, `Pop_Total`, `Pop_0_17`, `Pop_18_64`, `Pop_65_79`, `Pop_80_Plus`.

Treat `Municipality_Code` as an **identifier**, not a measurement.


In [ ]:
# ==============================================================================
# Load vintage, reference years, and the Module 1 anchor table
# ==============================================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

manifest_path = Path("data/manifest.json")
if not manifest_path.exists():
    raise FileNotFoundError(
        "data/manifest.json not found. Run this notebook from the course folder "
        "that contains the data/ directory (see C1DB1B2026_Data_Stream.pdf)."
    )

MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
m1 = MANIFEST["module1"]
T0 = int(m1["t0"])
T1 = int(m1["t1"])
anchor_path = Path(m1["anchor_file"])

print(f"Course vintage : {MANIFEST['vintage']}")
print(f"Anchor file    : {anchor_path}")
print(f"Reference years: t0 = {T0}, t1 = {T1}")

df_anchor = pd.read_csv(anchor_path)
print(f"Rows loaded    : {len(df_anchor)}")
print(f"Municipalities : {df_anchor['Municipality_Code'].nunique()}")
df_anchor.head(4)


---
## Task 1: State the decision and build the hierarchy

Write a short decision statement that identifies:
- Vårdlyftet as the client;
- the proposed primary care establishment;
- your five municipality alternatives (names and codes);
- the planning horizon (use the case horizon, or label a group assumption);
- the option of recommending that Vårdlyftet should **not** proceed.

Then construct:

**Decision Goal** → **Objectives** → **Criteria and Indicators**

Working definitions:
- **Objective:** an outcome Vårdlyftet wants to achieve.
- **Criterion:** an aspect used to judge the alternatives.
- **Indicator / KPI:** a defined measure used to represent a criterion.
- **Constraint:** a condition that must be satisfied, not traded off against a high score elsewhere.

Consider market opportunity, healthcare need, competition, accessibility, and financial or operational feasibility. You do not need a separate indicator for every subtopic.

**Self-check:** Could another group identify what you are deciding and how your proposed indicators connect to that decision?


---
### Your analysis for Task 1

**Decision statement:**
*(Write here.)*

**Five alternatives:**
| Code | Municipality |
| :--- | :--- |
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |

**Hierarchy (goal → objectives → criteria/indicators):**
*(Write here.)*

**Constraints (if any) and whether each threshold is a case requirement, a sourced requirement, or a group assumption:**
*(Write here.)*


---
## Task 2: Select your portfolio and run structural checks

Enter the five official SCB municipality codes allocated to your group.

Then keep only those codes and the two reference years $t_0$ and $t_1$ from the manifest. The core table must contain **ten rows** (5 municipalities $\times$ 2 years).

Required checks:
1. all five codes are present;
2. each municipality has one row for each reference year;
3. `Municipality_Code` and `Year` jointly identify rows uniquely;
4. required population fields are numeric, non-negative, and not missing;
5. the four age-group counts sum to `Pop_Total`.

If a check fails: identify the rows. Do **not** drop a municipality, replace missing values with zero, or edit the source file to force a pass. Raise a possible source-file problem with the instructor.


In [ ]:
# ==============================================================================
# TASK 2: Select five municipality codes and run structural checks
# ==============================================================================

# TODO: replace this example list with YOUR allocated codes
portfolio_codes = [1481, 1490, 1463, 1465, 1443]

value_fields = ["Pop_Total", "Pop_0_17", "Pop_18_64", "Pop_65_79", "Pop_80_Plus"]

df_portfolio = df_anchor[
    df_anchor["Municipality_Code"].isin(portfolio_codes)
    & df_anchor["Year"].isin([T0, T1])
].copy()

print("Selected rows:", len(df_portfolio))
print(df_portfolio.sort_values(["Municipality_Code", "Year"]).to_string(index=False))

# --- Structural checks ---
present = set(df_portfolio["Municipality_Code"].unique())
missing_codes = set(portfolio_codes) - present
print("\nCheck 1 missing codes:", missing_codes if missing_codes else "none")

counts = df_portfolio.groupby("Municipality_Code")["Year"].nunique()
print("Check 2 years per municipality:\n", counts.to_string())

dupes = df_portfolio.duplicated(subset=["Municipality_Code", "Year"]).sum()
print("Check 3 duplicate id pairs:", dupes)

numeric_ok = True
for col in value_fields:
    s = pd.to_numeric(df_portfolio[col], errors="coerce")
    if s.isna().any() or (s < 0).any():
        numeric_ok = False
        print(f"Check 4 FAIL on {col}")
if numeric_ok:
    print("Check 4 numeric / non-negative / non-missing: PASS")

band_sum = (
    df_portfolio["Pop_0_17"]
    + df_portfolio["Pop_18_64"]
    + df_portfolio["Pop_65_79"]
    + df_portfolio["Pop_80_Plus"]
)
mismatch = (band_sum != df_portfolio["Pop_Total"]).sum()
print("Check 5 age-band sum mismatches:", mismatch)


---
## Task 3: Calculate the common indicator set

Use the field definitions and years supplied with the anchor. Keep full precision during calculation. Round only displayed results. A zero denominator is **undefined**, not zero.

### Indicator 1 — Share of residents aged 80 and above (percent), year $t_1$
$$S_{80+,t_1} = 100\,\frac{P_{80+,t_1}}{P_{\mathrm{total},t_1}}$$

### Indicator 2 — Demographic dependency ratio, year $t_1$
$$D_{t_1} = 100\,\frac{P_{0-17,t_1}+P_{65-79,t_1}+P_{80+,t_1}}{P_{18-64,t_1}}$$
Unit: residents outside ages 18–64 per 100 residents aged 18–64. This is an age-structure measure, not employment.

### Indicator 3 — Growth in the population aged 80 and above between $t_0$ and $t_1$
$$G_{80+} = 100\,\frac{P_{80+,t_1}-P_{80+,t_0}}{P_{80+,t_0}}$$
Unit: percentage change over the specified period, **not** annual growth.

Do **not** calculate mean or median age from these four broad bands.

Produce one summary table with five rows. Then write two or three sentences: what the table reveals about your portfolio, and what it does **not** establish about clinic demand or business viability.


In [ ]:
# ==============================================================================
# TASK 3: Calculate S_80+, D, and G_80+ for each municipality
# ==============================================================================

df_t0 = df_portfolio[df_portfolio["Year"] == T0].set_index("Municipality_Code")
df_t1 = df_portfolio[df_portfolio["Year"] == T1].set_index("Municipality_Code")

summary = pd.DataFrame({
    "Municipality_Code": df_t1.index,
    "Municipality_Name": df_t1["Municipality_Name"].values,
    "Pop_Total_t1": df_t1["Pop_Total"].values,
    "Pop_80_Plus_t0": df_t0.loc[df_t1.index, "Pop_80_Plus"].values,
    "Pop_80_Plus_t1": df_t1["Pop_80_Plus"].values,
})

# TODO: complete the three indicators (the first is shown)
summary["S_80plus_t1"] = 100 * summary["Pop_80_Plus_t1"] / summary["Pop_Total_t1"]

# TODO: dependency ratio D_t1
# summary["D_t1"] = 100 * (
#     df_t1["Pop_0_17"] + df_t1["Pop_65_79"] + df_t1["Pop_80_Plus"]
# ) / df_t1["Pop_18_64"]
summary["D_t1"] = np.nan

# TODO: growth G_80plus
# summary["G_80plus"] = 100 * (
#     summary["Pop_80_Plus_t1"] - summary["Pop_80_Plus_t0"]
# ) / summary["Pop_80_Plus_t0"]
summary["G_80plus"] = np.nan

display_cols = [
    "Municipality_Code", "Municipality_Name",
    "S_80plus_t1", "D_t1", "G_80plus",
]
print(f"Prescribed indicators using t0={T0} and t1={T1}")
summary[display_cols].round(2)


---
### Interpretation of the summary table

*(Two or three sentences: what the table shows about age structure and 80+ change in your five municipalities, and what it does not show about clinic demand, competition, or financial viability.)*


---
## Task 4: Complete the KPI-to-data table

Propose **at least six distinct scored indicators**.

- At least **two** must be demonstrated using this anchor dataset.
- At least **one** competition or accessibility indicator.
- At least **one** financial or operational indicator.
- The rest are your group's choice. Those mapped indicators, once Demonstrated or Verified, are the columns Module 2 joins and Modules 3–4 score. Do not treat the three demographic calculations as the finished architecture.

For every scored indicator record:
1. criterion and relevance to the establishment decision;
2. definition or formula and unit;
3. preferred direction (benefit = higher is better; cost = lower is better) and why;
4. source and **exact fields**;
5. geographic level and reference period;
6. readiness and main limitation.

Readiness labels:
- **Demonstrated** — calculated from inspected data.
- **Verified** — fields and coverage inspected; full preparation deferred to Module 2.
- **Unresolved** — no usable measurement route yet (keep these in a separate information-gap list, not in the approved scored set).

A source name such as “SCB” without a table identifier or inspectable fields is **not** verification.


---
### KPI-to-data table (student template)

| # | Criterion | Indicator (formula / unit) | Direction | Source and exact fields | Geography / period | Readiness | Limitation |
| :---: | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | Healthcare need | $S_{80+,t_1}$ (%) | Benefit | Anchor: `Pop_80_Plus`, `Pop_Total` | Municipality / $t_1$ | Demonstrated | Age structure ≠ utilisation |
| 2 | Healthcare need | $G_{80+}$ (%) | Benefit | Anchor: `Pop_80_Plus` at $t_0$, $t_1$ | Municipality / $t_0$–$t_1$ | Demonstrated | Period change, not annual growth |
| 3 |  |  |  |  |  |  |  |
| 4 | Competition / accessibility |  |  |  |  | Verified or Unresolved |  |
| 5 | Financial / operational |  |  |  |  | Verified or Unresolved |  |
| 6 |  |  |  |  |  |  |  |

Rows 3–6 are **your** architecture, not a copy of the demonstration set. Name a real table identifier in the source column before you mark Verified. You will download those extracts before Module 2 and join them in `data/group_sources/`.

**Information-gap list (Unresolved only):**
*(Write here.)*


---
## Task 5: Record initial preferences

Assign a weight $w_j \geq 0$ to each indicator in the **proposed scored set**, with $\sum w_j = 1$.

State whose preferences the profile represents. If stakeholder preferences are assumed rather than supplied in the case, say so.

Weight the final scored indicators only. Do not also apply separate parent-objective weights in this exercise.

Record one consequential disagreement or assumption in your Part B process log (outside this notebook if you prefer).

Initial weights are provisional. Their consequences are examined later.


In [ ]:
# ==============================================================================
# TASK 5: Initial weights (must be non-negative and sum to 1)
# ==============================================================================

# TODO: replace names and weights with your approved scored set
weights = pd.Series({
    "S_80plus_t1": 0.25,
    "G_80plus": 0.20,
    "D_t1": 0.15,
    "clinics_per_10k": 0.15,
    "travel_time_min": 0.15,
    "premises_cost_sek": 0.10,
}, name="weight")

print("Sum of weights:", round(weights.sum(), 6))
print("Any negative?  ", bool((weights < 0).any()))
assert np.isclose(weights.sum(), 1.0, atol=1e-8), "Weights must sum to 1."
assert (weights >= 0).all(), "Weights must be non-negative."
weights.to_frame()


---
## Live sensitivity challenge (workshop)

The instructor will release a **controlled change** to the anchor fields during class. It uses the same formulas. It does not depend on your external datasets.

Your group must:
1. save pre-challenge results;
2. predict which indicators should change, and why;
3. apply the stated change to a **copy** of the baseline (never overwrite the CSV);
4. recalculate affected indicators;
5. produce a before-and-after comparison;
6. explain what changed, what did not, and why.

A stable result is not automatically better than a changed result. You must evaluate the change correctly.

Until the instructor releases the scenario, keep the cells below as a rehearsal using a documented example, or leave them commented.


In [ ]:
# ==============================================================================
# Live challenge rehearsal (replace with the instructor's exact instructions)
# Example only: raise Pop_80_Plus at t1 by 15% in the municipality with the
# smallest 80+ stock, then rebuild Pop_Total from the four age bands.
# ==============================================================================

baseline = summary[display_cols].copy()

df_challenge = df_portfolio.copy()
mask = (
    (df_challenge["Year"] == T1)
    & (df_challenge["Municipality_Code"] == int(summary.loc[summary["Pop_80_Plus_t1"].idxmin(), "Municipality_Code"]))
)
df_challenge.loc[mask, "Pop_80_Plus"] = (df_challenge.loc[mask, "Pop_80_Plus"] * 1.15).round(0)
df_challenge.loc[mask, "Pop_Total"] = (
    df_challenge.loc[mask, ["Pop_0_17", "Pop_18_64", "Pop_65_79", "Pop_80_Plus"]].sum(axis=1)
)

# TODO: recalculate S_80plus_t1, D_t1, G_80plus on df_challenge and compare with baseline
print("Baseline saved. Recalculate on df_challenge after the instructor release.")
baseline.round(2)


---
## Gateway 1 checklist

Submit one group package:
1. Framing brief (max two pages): decision statement, hierarchy, trade-offs, constraints, weight rationale.
2. KPI-to-data table: at least six scored indicators (not only the three common calculations), readiness evidence, initial weights, separate information-gap list. Named extracts for Verified indicators are ready to download before Module 2.
3. This notebook: structural checks, baseline indicator table, live-challenge comparison, short interpretations.

Do not duplicate the table or notebook inside the two-page brief.

**Method first, then your architecture.** Extra files do not replace the core method. They are required when they support an approved KPI.

---
*Documents:* A *The Assignment* (Canvas); B *The Data Stream* (`C1DB1B2026_Data_Stream.pdf`); module handbook (Canvas).
